In [1]:
import os
os.environ["NPU_VISIBLE_DEVICES"]="7"
os.environ["ASCEND_RT_VISIBLE_DEVICES"]="7"
import json
from tqdm import tqdm
from functools import partial
from typing import Optional, Tuple, Union, Dict, List, Any

import torch
import torch.nn as nn
from transformers import AutoTokenizer
from datasets import load_dataset, concatenate_datasets


from transformers.cache_utils import Cache
from transformers.utils import add_start_docstrings, ModelOutput, logging
from transformers.modeling_utils import PreTrainedModel
from transformers import LlamaModel, AutoConfig, AutoModelForCausalLM
from transformers.configuration_utils import PretrainedConfig

from typing import TYPE_CHECKING
from dataclasses import dataclass

logger = logging.get_logger(__name__)

if TYPE_CHECKING:
    from transformers import PreTrainedModel


/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/latest owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")
/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/8.0.RC2/aarch64-linux/ascend_toolkit_install.info owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")


In [2]:
base_model = "/data/pretrained-models/meta/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model)

In [3]:
base_dataset = "/data/datasets/Llama-3.2-3B-Instruct-evals"
general_datasets = [
    "/data/datasets/ultrachat_200k",
]
reason_datasets = [
    "allenai/cosmos_qa",
    "rajpurkar/squad_v2"
]
math_datasets = [
    "gsm8k",
    "/data/datasets/MathInstruct",
    "EleutherAI/hendrycks_math"
]
magicoder_datasets = [
    "/data/datasets/Magicoder-Evol-Instruct-110K",
]
leetcode_datasets = [
    "greengerong/leetcode"
]

In [4]:
def preprocess_gsm8k(examples:Dict[str, Any], tokenizer:AutoTokenizer)->Dict[str, Any]:
    prefix = "Given the following problem, reason and give a final answer to the problem.\nProblem: {{question}}\nYour response should end with \"The final answer is [answer]\" where [answer] is the response to the problem.\n"
    icl = [
        {
            "role" : "user",
            "content" : "There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?"
        },
        {
            "role" : "assistant",
            "content" : "There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The final answer is 6"
        },
        {
            "role": "user",
            "content": "If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?"
        },
        {
            "role": "assistant",
            "content" : "There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The final answer is 5"
        },
        {
            "role": "user",
            "content" : "Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?",
        },
        {
            "role" : "assistant",
            "content" : "Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. The final answer is 39"
        },
        {
            "role" : "user",
            "content" : "Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?"
        },
        {
            "role" : "assistant",
            "content" : "Jason started with 20 lollipops. Then he had 12 after giving some to Denny. So he gave Denny 20 - 12 = 8. The final answer is 8"
        },
        {
            "role" : "user",
            "content" : "Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?"
        },
        {
            "role" : "assistant",
            "content" : "Shawn started with 5 toys. If he got 2 toys each from his mom and dad, then that is 4 more toys. 5 + 4 = 9. The final answer is 9"
        },
        {
            "role" : "user",
            "content" : "There were nine computers in the server room. Five more computers were installed each day, from monday to thursday. How many computers are now in the server room?"
        },
        {
            "role" : "assistant",
            "content" : "There were originally 9 computers. For each of 4 days, 5 more computers were added. So 5 * 4 = 20 computers were added. 9 + 20 is 29. The final answer is 29"
        },
        {
            "role" : "user",
            "content" : "Michael had 58 golf balls. On tuesday, he lost 23 golf balls. On wednesday, he lost 2 more. How many golf balls did he have at the end of wednesday?"
        },
        {
            "role" : "assistant",
            "content" : "Michael started with 58 golf balls. After losing 23 on tuesday, he had 58 - 23 = 35. After losing 2 more, he had 35 - 2 = 33 golf balls. The final answer is 33"
        },
        {
            "role" : "user",
            "content" : "Olivia has $23. She bought five bagels for $3 each. How much money does she have left?"
        },
        {
            "role" : "assistant",
            "content" : "Olivia had 23 dollars. 5 bagels for 3 dollars each will be 5 x 3 = 15 dollars. So she has 23 - 15 dollars left. 23 - 15 is 8. The final answer is 8"
        }
    ]
    for i in range(len(icl)):
        if icl[i]['role'] == "user":
            icl[i]['content'] = prefix.replace("{{question}}", icl[i]['content'])
    return {
        "inst": prefix.replace("{{question}}", examples["question"].strip()),
        "response": examples["answer"].strip().replace("\n", " ").replace("#### ", "The final answer is "),
        "history": icl, 
        "system": ""
        }

def preprocess_math(examples:Dict[str, Any], tokenizer:AutoTokenizer) -> Dict[str, Any]:
    icl = [
        # {
        #     'role': 'user',
        #     'content': """Provide concise and precise solutions to the following math problems. Topics may include algebra, counting and probability, geometry, intermediate algebra, number theory, prealgebra, or precalculus. Show only essential steps and the final answer."""
        # },
        {
            'role': "user",
            'content': "A certain circle's area is $x$ square units, and its circumference is $y$ units. The value of $x + y$ is $80\pi$. What is the radius of the circle, in units?",
        },
        {
            "role": "assistant",
            "content": """If $r$ is the radius of the circle, we know that $x = \pi r^2$ and $y = 2\pi r$. Thus, we have \begin{align*}
x + y &= 80\pi\\
\pi r^2 + 2\pi r &= 80\pi\\
r(r + 2) &= 80.
\end{align*}

We want to find two integers whose product is 80, such that one integer is two more than the other. We note that 80 can be factored as $8\cdot 10$. Therefore, $r = \boxed{8}$.""",
        },
        {
            "role": "user",
            "content": "Consider two positive even integers less than $15$ (not necessarily distinct). When the sum of these two numbers is added to their product, how many different possible values may result?"
        },
        {
            "role": "assistant",
            "content": """Let $p$ and $q$ be the two integers; then $p,q \in \{2,4,6,8,10,12,14\}$, giving $7 \times 7 = 49$ total possible pairs $(p,q)$. The question asks for the number of different values of $pq + p + q$. Notice that by Simon's Favorite Factoring Trick, $$pq + p + q = (p+1)(q+1) - 1,$$so it suffices to find the number of different possible values of $(p+1)(q+1)$. Here, $p+1,q+1 \in \{3,5,7,9,11,13,15\}$.

There are $7$ pairs $(p,q)$ where $p+1$ is equal to $q+1$; by symmetry, half of the $42$ remaining pairs correspond to swapping the values of $p$ and $q$, leaving $42/2 = 21$ pairs $(p,q)$. Since most of the possible values of $p+1$ and $q+1$ are prime factors that do not divide into any of the other numbers, we note that most of the values of $(p+1)(q+1)$ will be distinct. The exception are the numbers divisible by $3$ and $5$: $p+1,q+1 \in \{3,5,9,15\}$; then, if $(p+1,q+1) = (3,15)$ or $(5,9)$, then $(p+1)(q+1) = 45$.

Hence, there are exactly $21-1+7 = \boxed{27}$ distinct possible values of $pq + p + q$."""
        },
        {
            "role": "user",
            "content": "A card is chosen at random from a standard deck of 52 cards, and then it is replaced and another card is chosen. What is the probability that at least one of the cards is a diamond or an ace?"
        },
        {
            "role": "assistant",
            "content": """There are 16 cards in a standard deck which are either diamonds or aces. The probability that neither card chosen is a diamond or an ace is $\left( \frac{36}{52} \right) ^2=\left( \frac{9}{13} \right) ^2=\frac{81}{169}$. Therefore, the probability that at least one of the cards chosen was a diamond or an ace is $1-\frac{81}{169}=\boxed{\frac{88}{169}}$."""
        },
        {
            "role": "user",
            "content": "What is the remainder when $1 + 2 + 3 + 4 + \dots + 9 + 10$ is  divided by 8?"
        },
        {
            "role": "assistant",
            "content": """Notice that we can pair many of these terms: \[1+7=2+6=3+5=8,\]so the remainder we want is the same as the remainder when $4+9+10$ is divided by 8.  We also see that this is the remainder when  \[4+1+2=7\]is divided by 8, so the answer is $\boxed{7}$."""
        },
        {
            "role": "user",
            "content": """The real numbers $a$ and $b$ satisfy
\[\begin{pmatrix} 2 \\ a \\ -7 \end{pmatrix} \times \begin{pmatrix} 5 \\ 4 \\ b \end{pmatrix} = \mathbf{0}.\]Enter the ordered pair $(a,b).$"""
        },
        {
            "role": "assistant",
            "content": """In general, $\mathbf{v} \times \mathbf{w} = \mathbf{0}$ if and only if the vectors $\mathbf{v}$ and $\mathbf{w}$ are proportional.  Thus, the vectors $\begin{pmatrix} 2 \\ a \\ -7 \end{pmatrix}$ and $\begin{pmatrix} 5 \\ 4 \\ b \end{pmatrix}$ are proportional.  Thus,
\[\frac{5}{2} = \frac{4}{a} = \frac{b}{-7}.\]Solving, we find $(a,b) = \boxed{\left( \frac{8}{5}, -\frac{35}{2} \right)}.$"""
        }
    ]
    return {
        'inst': examples['problem'],
        'response': examples['solution'],
        "history": icl,
        "system": """Provide concise and precise solutions to the following math problems. Topics may include algebra, counting and probability, geometry, intermediate algebra, number theory, prealgebra, or precalculus. Show only essential steps and the final answer."""
    }

def preprocess_cosmos(examples: Dict[str, Any], tokenizer:AutoTokenizer) -> Dict[str, Any]:
    prompt = """Given the context and question, select the most appropriate answer from the provided options. Then, repeat the full content of the selected option in your response.

Context:
Good Old War and person L: I saw both of these bands Wednesday night, and they both blew me away. Seriously. Good Old War is acoustic and makes me smile. I really can not help but be happy when I listen to them; I think it’s the fact that they seemed so happy themselves when they played.

Question:
In the future, will this person go to see other bands play?

A. None of the above choices.
B. This person likes music and likes to see the show, they will see other bands play.
C. This person only likes Good Old War and Person L, no other bands.
D. Other Bands is not on tour and this person cannot see them.

Answer:
A. None of the above choices."""
    prefix = "A"
    answer = 0
    if examples['label'] == 1:
        prefix = "B"
        answer = 1
    elif examples['label'] == 2:
        prefix = "C"
        answer = 2
    elif examples['label'] == 3:
        prefix = "D"
        answer = 3
    answer = f"answer{answer}"
    return {
        'inst': f"""Context:
{examples['context'].strip()}

Question:

{examples['question'].strip()}

A. {examples['answer0'].strip()}
B. {examples['answer1'].strip()}
C. {examples['answer2'].strip()}
D. {examples['answer3'].strip()}""",
        'response': f"{prefix}. {examples[answer].strip()}",
        "history": [],
        "system": prompt,
        # 'history': [
        #     {
        #         'role': 'user',
        #         'content': prompt
        #     }
        # ]
    }

def preprocess_sqaure(examples: Dict[str, Any], tokenizer:AutoTokenizer) -> Dict[str, Any]:
    prompt = """Given the context and the question, provide a concise, accurate answer in the format `[answer_start]: [text]`, where `answer_start` is the index of where the answer starts in the context, and `text` is the answer itself."""
    question = f"""Context:

{examples['context'].strip()}

Question:

{examples['question'].strip()}"""

    answers = [
        f"""{answer_start}: {txt.strip()}""" if txt.strip()[-1] == '.'
        else f"""{answer_start}: {txt.strip()}."""
        for txt, answer_start in zip(examples['answers']['text'], examples['answers']['answer_start'])
    ]
    if len(answers) == 0:
        answers_ = "-1: No answer founded."
    else:
        answers_ = answers[0]
        for item in answers[1:]:
            answers += f"\n{item}"
    return {
        "inst": question,
        "response": answers_,
        "history": [
            # {
            #     "role": "user",
            #     "content": prompt
            # },
            {
                "role": "user",
                "content": """Context:

The most widely spoken family of languages in southern Europe are the Romance languages, the heirs of Latin, which have spread from the Italian peninsula, and are emblematic of Southwestern Europe. (See the Latin Arch.) By far the most common romance languages in Southern Europe are: Italian, which is spoken by over 50 million people in Italy, San Marino, and the Vatican; and Spanish, which is spoken by over 40 million people in Spain and Gibraltar. Other common romance languages include: Romanian, which is spoken in Romania and Moldova; Portuguese, which is spoken in Portugal; Catalan, which is spoken in eastern Spain; and Galician, which is spoken in northwestern Spain.

Question:

What are the three main areas of southern Europe where Italian speakers can be found?"""
            },
            {
                "role": "assistant",
"content": """339: Italy, San Marino, and the Vatican."""
            }
        ],
        "system": prompt,
    }

def preprocess_llama3_eval(examples:Dict[str, Any], tokenizer:AutoTokenizer) -> Dict[str, Any]:
    input_final_prompts = examples['input_final_prompts'][0].replace("<|eot_id|>", "")
    input_final_prompts = input_final_prompts.replace("<|start_header_id|>user<|end_header_id|>", "#$%2$%$#")
    input_final_prompts = input_final_prompts.split("#$%2$%$#")[1:]
    input_final_prompts = [item.split("<|start_header_id|>assistant<|end_header_id|>") for item in input_final_prompts]
    history = []
    for question, answer in input_final_prompts[-1]:
        history.append({"role": "user", "content": question})
        history.append({"role": "assistant", "content": answer})
    return {
        "inst": input_final_prompts[-1][0],
        "response": examples['output_prediction_text'][0],
        "history": history,
        "system": ""
    }

def preprocess_ultrachat(examples:Dict[str, Any], tokenizer:AutoTokenizer)->Dict[str,Any]:
    messages = examples['messages']

    return { 
        "inst": messages[-2]['content'],
        "response": messages[-1]['content'],
        "history": messages[:-2] if len(messages) > 2 else [],
        "system": "",
    }

In [5]:
def task_preprocess(example:Dict[str, str], tokenizer:AutoTokenizer, task:str="humaneval")->Dict[str, str]:
    if task == "humaneval":
        instruction_prefix = "Please provide a self-contained Python script that solves the following problem in a markdown code block:"
        response_prefix = "Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:"
        # some random words which servcleaes as the splitter
        _MAGIC_SPLITTER_ = "-[[]]-this-is-really-our-highest-priority-[[]]-"
        task_prompt = f"""\
{instruction_prefix}
```
{example['prompt'].strip()}
```
"""
        response = f"""\
{response_prefix}
```python
{example}
```
"""
        task_prompt = tokenizer.apply_chat_template(
            [
                {"role": "user", "content": task_prompt},
                {"role": "assistant", "content": response},
            ],
            tokenize=False,
        ).split(_MAGIC_SPLITTER_)[0]
        return {
            "inst": task_prompt,
        }
    elif task == "mbpp":
        instruction_prefix = "Please provide a self-contained Python script that solves the following problem in a markdown code block:"
        response_prefix = "Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:"
        # some random words which servcleaes as the splitter
        _MAGIC_SPLITTER_ = "-[[]]-this-is-really-our-highest-priority-[[]]-"
        python_prefix = 'Write a python function to '
        func_prefix = 'Write a function to '
        if python_prefix in example['prompt']:
            prefix = python_prefix
        elif func_prefix in example['prompt']:
            prefix = func_prefix
        else:
            prefix = ""
        prompt = example['prompt'].replace(prefix, '').strip().capitalize()
        task_prompt = f"""\
{instruction_prefix}
```
{example['code'].split(":")[0].strip()}:
    \"\"\"
    {prompt}
    >>> {example['test_list'][0].replace("assert", "").strip()}
    True
    \"\"\"
```
"""
        response = f"""\
{response_prefix}
```python
{_MAGIC_SPLITTER_}
```
"""
        task_prompt = tokenizer.apply_chat_template(
            [
                {"role": "user", "content": task_prompt},
                {"role": "assistant", "content": response},
            ],
            tokenize=False,
        ).split(_MAGIC_SPLITTER_)[0]
        return {
            "inst": task_prompt,
        }
    elif task == "magicoder":
        return {
            "inst": example['instruction'].strip(),
            "response": example['response'].strip(),
            "history": [],
            "system": "",
        }
    elif task == "mathinstruct":
        return {
            "inst": example['instruction'].strip(),
            "response": example['output'].strip(),
            "history": [],
            "system": "",
        }
    elif task == "leetcode":
        content = [item.strip() for item in example['content']]
        java = [item.strip() for item in example['java']]
        python = [item.strip() for item in example['python']]
        cpp = [item.strip() for item in example['c++']]
        javascript = [item.strip() for item in example['javascript']]
        inst, response, history, system = [], [], [], []
        for item in zip(content, java, python, cpp, javascript):
            inst.extend([item[0]] * len(item[1:]))
            response.extend(item[1:])
            history.extend([[] for _ in range(len(item[1:]))])
            system.extend([""] * len(item[1:]))
        return {
            "inst": inst,
            "response": response,
            "history": history,
            "system": system
        }

In [6]:
general_datasets = [
    load_dataset(
        general_datasets[0],
        num_proc=8,
    )['train_sft'],
]
general_datasets = [
    general_datasets[0].filter(
        lambda x: len(x['messages']) > 1,
    ).map(
        partial(preprocess_ultrachat, tokenizer=tokenizer),
        num_proc=8
    )
]
reason_datasets = [
    load_dataset(
        reason_datasets[0],
        num_proc=8,
    )['train'].map(
        partial(preprocess_cosmos, tokenizer=tokenizer),
        num_proc=8,
    ),
    load_dataset(
        reason_datasets[1],
        num_proc=8,
    )['train'].map(
        partial(preprocess_sqaure, tokenizer=tokenizer),
        num_proc=8,
    ),
]

math_config = [
    'algebra',
    'counting_and_probability',
    'geometry',
    'intermediate_algebra',
    'number_theory',
    'prealgebra',
    'precalculus'
]

math_datasets = [
    load_dataset(
        math_datasets[0],
        "main",
        num_proc=8,
    )['train'].map(
        partial(preprocess_gsm8k, tokenizer=tokenizer),
        num_proc=8,
    ),
    load_dataset(
        math_datasets[1],
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="mathinstruct"),
        num_proc=8,
    )['train'],
    concatenate_datasets([
        load_dataset(
            math_datasets[2],
            item,
            num_proc=8
        )['train'] for item in math_config
    ]).map(
        partial(preprocess_math, tokenizer=tokenizer),
        num_proc=8,
    )
]

magicoder_datasets = [
    load_dataset(
        item,
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="magicoder"),
        num_proc=8,
        # load_from_cache_file=False,
    )['train'] for item in magicoder_datasets
]

leetcode_datasets = [
    load_dataset(
        item,
        num_proc=8,
    )['train'] for item in leetcode_datasets
]
leetcode_datasets = [
    item.map(
        partial(task_preprocess, tokenizer=tokenizer, task="leetcode"),
        num_proc=8,
        batched=True,
        remove_columns=item.column_names
    )
    for item in leetcode_datasets
]

Using the latest cached version of the module from /data/lihz/.cache/huggingface/modules/datasets_modules/datasets/allenai--cosmos_qa/3e18538cbfdb2c04189b16642715f0f6da3e97ed5df0aadcec3641245b2cf157 (last modified on Sun Dec 22 17:36:47 2024) since it couldn't be found locally at allenai/cosmos_qa, or remotely on the Hugging Face Hub.


In [7]:
print(len(general_datasets[0]),
      len(reason_datasets[0]),
      len(reason_datasets[1]),
      len(math_datasets[0]),
      len(math_datasets[1]),
      len(math_datasets[2]),
      len(leetcode_datasets[0]),
      len(magicoder_datasets[0]))

207865 25262 130319 7473 262039 7500 9440 111183


In [8]:
mix_domains = []

start = 4100
samples = 5000


repeat = 1 if len(general_datasets[0]) >= samples else 0
for _ in range(repeat):
    for i, item in enumerate(general_datasets[0]):
        if i <= start + 200:
            continue
        if i == samples + 200:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'ultrachat',
            'task_label': 0,
            'label': 0,
            'outputs': item['response'],
            'history': item['history'],
            'system': item['system'],
        })

In [9]:
repeat = 1 if len(reason_datasets[0]) >= samples else 0
for _ in range(repeat):
    for i, item in enumerate(reason_datasets[0]):
        if i <= start:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'cosmos',
            'task_label': 1,
            'label': 1,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })
repeat = 1 if len(reason_datasets[1]) >= samples else 1
for _ in range(repeat):
    for i, item in enumerate(reason_datasets[1]):
        if i <= start:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'square',
            'task_label': 2,
            'label': 1,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })


In [10]:
repeat = 1 if len(math_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(math_datasets[0]):
        if i <= start:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'gsm8k',
            'task_label': 3,
            'label': 2,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })

repeat = 1 if len(math_datasets[1]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(math_datasets[1]):
        if i <= start:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'mathinstruct',
            'task_label': 4,
            'label': 2,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })

repeat = 1 if len(math_datasets[2]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(math_datasets[2]):
        if i <= start:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'math',
            'task_label': 5,
            'label': 2,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })

In [11]:

repeat = 1 if len(leetcode_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(leetcode_datasets[0]):
        if i <= start:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'leetcode',
            'task_label': 6,
            'label': 3,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })

repeat = 1 if len(magicoder_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(magicoder_datasets[0]):
        if i <= start + 500:
            continue
        if i == samples + 500:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'magicoder',
            'task_label': 7,
            'label': 3,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })

In [12]:
print(len(mix_domains))

7192


In [13]:
# with open("/data/lihz/datasets/mix_domains_eval/mix_domains_eval_x1.jsonl", 'w') as f:
#     for item in mix_domains:
#         f.write(json.dumps(item) + '\n')

In [14]:
mix_domains_tokens = []
for item in tqdm(mix_domains):
    prompts = []
    prompts.append({
        "role": "system",
        "content": item['system']
    })
    prompts.extend(item['history'])
    prompts.append({
        'role': 'user',
        'content': item['inputs']
    })
    input_ids = tokenizer.apply_chat_template(prompts, add_generation_prompt=True)
    prompts.append({
        'role': 'assistant',
        'content': item['outputs']
    })
    whole_input_ids = tokenizer.apply_chat_template(prompts)
    labels = len(input_ids) * [-100] + whole_input_ids[len(input_ids):]
    mix_domains_tokens.append({
        'input_ids': whole_input_ids,
        'labels': labels,
        'task': item['task_label'],
        'type': item['label']
    })


100%|██████████| 7192/7192 [00:38<00:00, 184.44it/s]


In [15]:
tag=1600
# print(mix_domains_tokens[tag])
# print(mix_domains_tokens[tag]['input_ids'])
print(mix_domains_tokens[tag]['labels'])
print(mix_domains[tag]['outputs'])
print(tokenizer.decode(mix_domains_tokens[tag]['input_ids']))


[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -10

In [16]:


@dataclass
class CLSOutput(ModelOutput):
    loss: Optional[Union[torch.FloatTensor, Dict[str, torch.FloatTensor]]] = None
    hidden_states: Optional[Union[Tuple[torch.FloatTensor, ...], Dict[str, torch.FloatTensor]]] = None
    activations: Optional[Union[Tuple[torch.FloatTensor, ...], Dict[str, torch.FloatTensor]]] = None

CLS_START_DOCSTRING = r"""
    This model inherits from [`PreTrainedModel`]. Check the superclass documentation for the generic methods the
    library implements for all its model (such as downloading or saving, resizing the input embeddings, pruning heads
    etc.)

    This model is also a PyTorch [torch.nn.Module](https://pytorch.org/docs/stable/nn.html#torch.nn.Module) subclass.
    Use it as a regular PyTorch Module and refer to the PyTorch documentation for all matter related to general usage
    and behavior.

    Parameters:
        config ([`CLSConfig`]):
            Model configuration class with all the parameters of the model. Initializing with a config file does not
            load the weights associated with the model, only the configuration. Check out the
            [`~PreTrainedModel.from_pretrained`] method to load the model weights.
"""

class CLSConfig(PretrainedConfig):
    r"""
    This is the configuration class to store the configuration of a [`CLSModel`]. It is used to instantiate an CLS
    model according to the specified arguments, defining the model architecture. Instantiating a configuration with the
    defaults will yield a similar configuration to that of the CLS-7B.

    Configuration objects inherit from [`PretrainedConfig`] and can be used to control the model outputs. Read the
    documentation from [`PretrainedConfig`] for more information.


    Args:
        vocab_size (`int`, *optional*, defaults to 32000):
            Vocabulary size of the CLS model. Defines the number of different tokens that can be represented by the
            `inputs_ids` passed when calling [`CLSModel`]
        hidden_size (`int`, *optional*, defaults to 4096):
            Dimension of the hidden representations.
        num_hidden_layers (`int`, *optional*, defaults to 32):
            Number of hidden layers in the Transformer decoder.
        hidden_act (`str` or `function`, *optional*, defaults to `"silu"`):
            The non-linear activation function (function or string) in the decoder.
        initializer_range (`float`, *optional*, defaults to 0.02):
            The standard deviation of the truncated_normal_initializer for initializing all weight matrices.

    ```python
    >>> from transformers import CLSModel, CLSConfig

    >>> # Initializing a CLS CLS-7b style configuration
    >>> configuration = CLSConfig()

    >>> # Initializing a model from the CLS-7b style configuration
    >>> model = CLSModel(configuration)

    >>> # Accessing the model configuration
    >>> configuration = model.config
    ```"""

    model_type = "cls"

    def __init__(
        self,
        hidden_size:int=4096,
        num_labels:int=4,
        num_tasks:int=8,
        # num_block:int=4,
        label_block:int=4,
        task_block:int=4,
        initializer_range:float=0.02,
        **kwargs,
    ):
        super().__init__(
            **kwargs,
        )
        self.num_labels= num_labels
        self.num_tasks = num_tasks
        # self.num_block = num_block
        self.label_block = label_block
        self.task_block = task_block
        self.hidden_size = hidden_size
        self.initializer_range = initializer_range

@add_start_docstrings(
    "The bare CLS Model outputting raw hidden-states without any specific head on top.",
    CLS_START_DOCSTRING,
)
class CLSPreTrainedModel(PreTrainedModel):
    config_class = CLSConfig
    base_model_prefix = "model"

    def _init_weights(self, module):
        std = self.config.initializer_range
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=std)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=std)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()

class CLS(CLSPreTrainedModel):
    def __init__(
            self, 
            config: CLSConfig,
            **kwargs
            ):
        super().__init__(config)
        # self.model_config = AutoConfig.from_pretrained("/data/pretrained-models/meta/Llama-3.2-3B-Instruct")
        self.model = LlamaModel.from_pretrained("/data/pretrained-models/meta/Llama-3.2-3B-Instruct")
        # self.model = AutoModelForCausalLM.from_pretrained("/data/pretrained-models/meta/Llama-3.2-3B-Instruct")
        self.model.return_hidden_states = True
        self.model.requires_grad_(False)
        self.model.eval()
        self.num_tasks = config.num_tasks
        self.num_labels = config.num_labels
        # self.num_block = config.num_block
        self.label_block = config.label_block
        self.task_block = config.task_block
        self.task_score = nn.Linear(config.hidden_size, 
                                    config.num_tasks * self.task_block, 
                                    bias=False, 
                                    device=self.model.device, 
                                    dtype=self.model.dtype)
        self.label_score = nn.Linear(config.hidden_size, 
                                     config.num_labels * self.label_block, 
                                     bias=False, 
                                     device=self.model.device, 
                                     dtype=self.model.dtype)

        self.num_labels = config.num_labels
        self.num_tasks = config.num_tasks
        self.ignore_index = -100
        self.post_init()
        
        self.smooth = 0.75
        self.label_mask = nn.Parameter(
            torch.kron(
                torch.ones([self.label_block, self.label_block]), 
                torch.eye(self.num_labels) * self.smooth) + torch.eye(config.num_labels * self.label_block) * (1 - self.smooth))
        self.task_mask = nn.Parameter(
            torch.kron(
                torch.ones([self.task_block, self.task_block]), 
                torch.eye(self.num_tasks) * self.smooth) + torch.eye(config.num_tasks * self.task_block) * (1 - self.smooth))

    def forward(
        self,
        input_ids: torch.LongTensor = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values: Optional[Union[Cache, List[torch.FloatTensor]]] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        labels: Optional[torch.LongTensor] = None,
        type_labels: Optional[torch.LongTensor] = None,
        task_labels: Optional[torch.LongTensor] = None,
        use_cache: Optional[bool] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        cache_position: Optional[torch.LongTensor] = None,
    ) -> Union[Dict, Tuple, torch.Tensor, CLSOutput]:
        with torch.no_grad():
            outputs = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                position_ids=position_ids,
                past_key_values=past_key_values,
                inputs_embeds=inputs_embeds,
                # labels=labels,
                use_cache=use_cache,
                output_attentions=output_attentions,
                output_hidden_states=output_hidden_states,
                return_dict=return_dict,
                cache_position=cache_position,
            )
            # hidden_states = outputs[1]
            hidden_states = outputs[0]

        def norm(x:torch.Tensor, eps:torch.Tensor=1e-8) -> torch.Tensor:
            dtype = x.dtype
            x = x.float()
            x = x - x.mean(-1, keepdim=True)
            x = x * (x.pow(2).mean(-1, keepdim=True) + eps).rsqrt()
            return x.to(dtype)
        
        def proc(x:torch.Tensor, m:torch.Tensor, eps:torch.Tensor=1e-8) -> torch.Tensor:
            dtype = x.dtype
            x = x.float()
            x = x.sum(dim=1) / (m.sum(dim=1, keepdim=True) + eps)
            return x.to(dtype)

        shift_hidden_states = hidden_states[..., :-1, :]
        shift_label_masks = labels[..., 1:].not_equal(-100)
        
        shift_hidden_states = norm(shift_hidden_states)
        shift_hidden_states = shift_hidden_states * shift_label_masks.unsqueeze(-1)
        shift_hidden_states = proc(shift_hidden_states, shift_label_masks)
        
        label_scores = self.label_score(shift_hidden_states).view(-1, self.num_labels).abs().float()
        task_scores = self.task_score(shift_hidden_states).view(-1, self.num_tasks).abs().float()

        label_logits = torch.softmax(label_scores, dim=-1)
        task_logits = torch.softmax(task_scores, dim=-1)

        if type_labels is not None and task_labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            return (label_logits, 
                    task_logits, 
                    loss_fct(label_scores, type_labels.repeat_interleave(self.label_block, 0)), 
                    loss_fct(task_scores, task_labels.repeat_interleave(self.task_block, 0)), )

        return (label_logits, task_logits, )
        

In [17]:
# model:CLS = CLS.from_pretrained("/data/lihz/projects/instruct/IFT/ift/cls_48_v5")
model:CLS = CLS.from_pretrained("/data/lihz/projects/instruct/IFT/ift/cls_48_x1")
# model.model = LlamaModel.from_pretrained("/data/lihz/projects/instruct/IFT/code/3b-e2")
# model.model = LlamaModel.from_pretrained("/data/lihz/projects/instruct/IFT/ift/wcls_48_v2/models")
# model.model = LlamaModel.from_pretrained("/data/lihz/projects/instruct/IFT/ift/wcls_48_v4/models")
# model.model = LlamaModel.from_pretrained("/data/lihz/projects/instruct/IFT/ift/wcls_48_v5/models")
model.model = LlamaModel.from_pretrained("/data/lihz/projects/instruct/IFT/ift/rwcls_48_v6/models")
# model = CLS.from_pretrained("/data/lihz/projects/instruct/IFT/ift/wcls_48_v4")
# model.model = AutoModelForCausalLM.from_pretrained("/data/pretrained-models/meta/Llama-3.2-3B-Instruct")
# model.model.return_hidden_states = True
# model = CLS(CLSConfig.from_pretrained("/data/lihz/projects/instruct/IFT/ift/wcls_48_v4"))
model = model.to("npu:0")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [18]:
# from safetensors.torch import load_file
# x = load_file("/data/lihz/projects/instruct/IFT/ift/wcls_48_v4/model-00002-of-00002.safetensors")
# print(x.keys())
# print(x['label_score.weight'])
# from safetensors.torch import load_file
# x = load_file("/data/lihz/projects/instruct/IFT/ift/cls_48_v6/model-00002-of-00002.safetensors")
# print(x.keys())
# print(x['label_score.weight'])

In [19]:
x = model.label_score.weight
print(x.shape)
x = x.float()
s = x.pow(2).sum(-1).rsqrt()
y:torch.Tensor = torch.matmul(x, x.transpose(-1, -2)) * s.unsqueeze(1) * s.unsqueeze(0)
print(y.shape)
# print(y.detach().cpu().numpy())
print(y.fill_diagonal_(0.0).pow(2).sum())

torch.Size([3072, 3072])
torch.Size([3072, 3072])
tensor(2356.8738, device='npu:0', grad_fn=<SumBackward0>)


In [20]:
# print(model.label_score.weight.shape)
# print(model.label_score.weight)

In [21]:
# Parameter containing:
# tensor([[ 0.0049, -0.0068, -0.0153,  ...,  0.0143,  0.0045,  0.0139],
#         [-0.0219,  0.0062,  0.0173,  ...,  0.0173,  0.0161, -0.0234],
#         [ 0.0149,  0.0121,  0.0330,  ..., -0.0183,  0.0212, -0.0197],
#         ...,
#         [ 0.0145, -0.0244, -0.0021,  ...,  0.0332, -0.0076,  0.0157],
#         [ 0.0188,  0.0369,  0.0011,  ...,  0.0623,  0.0043,  0.0253],
#         [ 0.0194,  0.0271, -0.0267,  ...,  0.0025, -0.0216,  0.0139]],
#        device='npu:0', requires_grad=True)

In [22]:
x = model.task_score.weight
print(x.shape)
x = x.float()
s = x.pow(2).sum(-1).rsqrt()
y:torch.Tensor = torch.matmul(x, x.transpose(-1, -2)) * s.unsqueeze(1) * s.unsqueeze(0)
print(y.fill_diagonal_(0.0).pow(2).sum())

torch.Size([3072, 3072])
tensor(2504.5615, device='npu:0', grad_fn=<SumBackward0>)


In [23]:
# 100 1000 2000 3000 4000 5000 6000 7000
tag = 5000
with torch.no_grad():
    outputs = model(
        input_ids=torch.tensor([mix_domains_tokens[tag]['input_ids']], device=model.device),
        labels=torch.tensor([mix_domains_tokens[tag]['labels']], device=model.device),
        type_labels=torch.tensor([mix_domains_tokens[tag]['type']], device=model.device),
        task_labels=torch.tensor([mix_domains_tokens[tag]['task']], device=model.device),
        use_cache=False,
    )
# print(outputs[0].cpu())
# print(outputs[1].cpu())
# print(outputs[2].cpu().sum())
# print(outputs[3].cpu())
# print(tokenizer.decode(mix_domains_tokens[tag]['input_ids']))
# print(mix_domains[tag]['input_ids'])
# print(mix_domains_tokens[tag]['type'])
# print(mix_domains_tokens[tag]['task'])

In [24]:
# tensor([[ 2.2861e+00, -1.8679e+00, -4.3208e+00, -6.9433e+00],
#         [ 1.3611e+00, -1.4691e+00, -4.0150e+00, -6.8870e+00],
#         [-2.3027e+00, -1.1429e+00, -5.5198e+00, -6.2148e+00],
#         [-2.7822e+00, -6.3820e-03, -3.0781e+00, -7.3422e+00]], device='npu:0')
# tensor([[2.2861e+00, 1.8679e+00, 4.3208e+00, 6.9433e+00],
#         [1.3611e+00, 1.4691e+00, 4.0150e+00, 6.8869e+00],
#         [2.3027e+00, 1.1429e+00, 5.5198e+00, 6.2148e+00],
#         [2.7822e+00, 6.3820e-03, 3.0781e+00, 7.3422e+00]], device='npu:0')

# tensor([[-0.8961,  0.7803, -8.9838, -3.7413],
#         [-1.0620,  1.1654, -9.3980, -3.0482],
#         [ 0.1878, -0.4052, -7.3323, -1.8446],
#         [ 0.8277, -0.9281, -9.0326, -3.5161]], device='npu:0')
# tensor([[0.8961, 0.7803, 8.9838, 3.7413],
#         [1.0620, 1.1654, 9.3980, 3.0482],
#         [0.1878, 0.4052, 7.3323, 1.8446],
#         [0.8277, 0.9281, 9.0326, 3.5161]], device='npu:0')

In [25]:
# tensor([[-0.5305, -0.1513,  2.1991,  ...,  0.1707,  0.0481,  0.1257]],
#        device='npu:0')
# tensor([[ 0.0171,  0.0012, -0.0021,  ...,  0.0022, -0.0014,  0.0001],
#         [ 0.0139, -0.0016, -0.0443,  ...,  0.0057,  0.0002,  0.0023],
#         [-0.0096,  0.0031,  0.0416,  ..., -0.0044, -0.0007, -0.0029],
#         ...,
#         [ 0.0079, -0.0075,  0.0462,  ...,  0.0062, -0.0012, -0.0014],
#         [-0.0021, -0.0054,  0.0294,  ..., -0.0025,  0.0009,  0.0046],
#         [ 0.0016,  0.0023, -0.0354,  ..., -0.0055,  0.0005, -0.0007]],
#        device='npu:0')
# tensor([ 2.8819, -3.7100, -2.6751,  0.1953,  2.9789, -3.9457, -1.6979, -0.4220,
#          3.3594, -3.0724, -1.6223,  0.0524,  2.7875, -2.8276, -2.5134, -0.1559],
#        device='npu:0')

# tensor([[-0.0749, -0.2556,  0.3035,  ...,  0.4693,  0.0854, -0.6732]],
#        device='npu:0')
# tensor([[ 0.0024,  0.0019, -0.0003,  ...,  0.0061, -0.0025, -0.0007],
#         [ 0.0020, -0.0027, -0.0061,  ...,  0.0156,  0.0004, -0.0126],
#         [-0.0014,  0.0053,  0.0057,  ..., -0.0120, -0.0013,  0.0155],
#         ...,
#         [ 0.0011, -0.0126,  0.0064,  ...,  0.0171, -0.0021,  0.0077],
#         [-0.0003, -0.0092,  0.0041,  ..., -0.0069,  0.0016, -0.0247],
#         [ 0.0002,  0.0039, -0.0049,  ..., -0.0152,  0.0009,  0.0039]],
#        device='npu:0')
# tensor([-3.3917,  7.6074, -4.3535, -3.6436, -2.1388,  8.4936, -2.9636, -2.7647,
#         -2.8613,  7.5219, -3.8216, -4.1020, -3.9688,  6.8251, -4.2285, -4.0234],
#        device='npu:0')

# tensor([[ 0.5567, -0.2362, -0.4617,  ...,  0.0302, -0.6205, -0.4817]],
#        device='npu:0')
# tensor([[-0.0179,  0.0018,  0.0004,  ...,  0.0004,  0.0179, -0.0005],
#         [-0.0145, -0.0025,  0.0093,  ...,  0.0010, -0.0031, -0.0090],
#         [ 0.0101,  0.0049, -0.0087,  ..., -0.0008,  0.0095,  0.0111],
#         ...,
#         [-0.0083, -0.0116, -0.0097,  ...,  0.0011,  0.0151,  0.0055],
#         [ 0.0022, -0.0085, -0.0062,  ..., -0.0004, -0.0117, -0.0176],
#         [-0.0017,  0.0036,  0.0074,  ..., -0.0010, -0.0067,  0.0028]],
#        device='npu:0')
# tensor([-3.7740,  8.1903, -2.8369, -3.4400, -3.9131,  6.6488, -4.3539, -4.1257,
#         -4.4823,  6.5965, -4.3207, -4.5006, -4.1457,  6.6505, -5.0347, -4.4331],
#        device='npu:0')

# tensor([[ 0.1926,  0.3491,  1.1178,  ..., -1.0553, -0.0436, -0.0511]],
#        device='npu:0')
# tensor([[-6.2058e-03, -2.6636e-03, -1.0661e-03,  ..., -1.3720e-02,
#           1.2570e-03, -4.9742e-05],
#         [-5.0304e-03,  3.7290e-03, -2.2515e-02,  ..., -3.5040e-02,
#          -2.1837e-04, -9.5505e-04],
#         [ 3.5025e-03, -7.2449e-03,  2.1151e-02,  ...,  2.7053e-02,
#           6.6843e-04,  1.1735e-03],
#         ...,
#         [-2.8678e-03,  1.7217e-02,  2.3470e-02,  ..., -3.8389e-02,
#           1.0652e-03,  5.8676e-04],
#         [ 7.5809e-04,  1.2529e-02,  1.4942e-02,  ...,  1.5588e-02,
#          -8.2556e-04, -1.8727e-03],
#         [-5.9648e-04, -5.3484e-03, -1.8012e-02,  ...,  3.4267e-02,
#          -4.7137e-04,  2.9494e-04]], device='npu:0')
# tensor([-2.8060, -2.1047,  5.0870, -1.9059, -2.9825, -2.6807,  4.0743, -3.1306,
#         -2.2165, -2.1211,  5.0250, -1.8568, -3.3235, -2.7455,  4.5207, -2.2509],
#        device='npu:0')

# tensor([[-0.3608, -0.4393,  0.9635,  ..., -0.8200, -0.5976,  0.0731]],
#        device='npu:0')
# tensor([[ 1.1626e-02,  3.3517e-03, -9.1889e-04,  ..., -1.0661e-02,
#           1.7217e-02,  7.1134e-05],
#         [ 9.4245e-03, -4.6923e-03, -1.9407e-02,  ..., -2.7228e-02,
#          -2.9911e-03,  1.3658e-03],
#         [-6.5619e-03,  9.1166e-03,  1.8231e-02,  ...,  2.1021e-02,
#           9.1557e-03, -1.6782e-03],
#         ...,
#         [ 5.3728e-03, -2.1665e-02,  2.0230e-02,  ..., -2.9830e-02,
#           1.4591e-02, -8.3910e-04],
#         [-1.4203e-03, -1.5766e-02,  1.2879e-02,  ...,  1.2112e-02,
#          -1.1308e-02,  2.6780e-03],
#         [ 1.1175e-03,  6.7302e-03, -1.5526e-02,  ...,  2.6627e-02,
#          -6.4564e-03, -4.2178e-04]], device='npu:0')
# tensor([-1.3215, -2.9045,  2.4800, -1.2478, -1.8010, -3.5329,  2.3942, -0.3438,
#         -0.8685, -2.4487,  2.1788, -0.5738, -1.5508, -3.0367,  2.3103, -0.7527],
#        device='npu:0')

# tensor([[-0.2302,  0.8603,  1.2694,  ..., -0.8639, -0.8852, -0.3156]],
#        device='npu:0')
# tensor([[ 0.0074, -0.0066, -0.0012,  ..., -0.0112,  0.0255, -0.0003],
#         [ 0.0060,  0.0092, -0.0256,  ..., -0.0287, -0.0044, -0.0059],
#         [-0.0042, -0.0179,  0.0240,  ...,  0.0221,  0.0136,  0.0072],
#         ...,
#         [ 0.0034,  0.0424,  0.0267,  ..., -0.0314,  0.0216,  0.0036],
#         [-0.0009,  0.0309,  0.0170,  ...,  0.0128, -0.0167, -0.0116],
#         [ 0.0007, -0.0132, -0.0205,  ...,  0.0281, -0.0096,  0.0018]],
#        device='npu:0')
# tensor([-3.7066, -4.2427,  5.3872, -0.4614, -3.8359, -4.0321,  4.5175, -0.1953,
#         -2.9209, -4.2535,  5.5962, -0.5347, -3.7497, -3.8572,  5.0464, -0.5227],
#        device='npu:0')

# tensor([[-0.2266,  0.1247,  0.8434,  ..., -0.1366, -0.1701,  0.1046]],
#        device='npu:0')
# tensor([[ 0.0073, -0.0010, -0.0008,  ..., -0.0018,  0.0049,  0.0001],
#         [ 0.0059,  0.0013, -0.0170,  ..., -0.0045, -0.0009,  0.0020],
#         [-0.0041, -0.0026,  0.0160,  ...,  0.0035,  0.0026, -0.0024],
#         ...,
#         [ 0.0034,  0.0062,  0.0177,  ..., -0.0050,  0.0042, -0.0012],
#         [-0.0009,  0.0045,  0.0113,  ...,  0.0020, -0.0032,  0.0038],
#         [ 0.0007, -0.0019, -0.0136,  ...,  0.0044, -0.0018, -0.0006]],
#        device='npu:0')
# tensor([-3.1639, -3.3484,  0.0416,  4.1010, -3.0149, -4.1082,  0.2010,  4.5231,
#         -3.5141, -4.0539,  1.0198,  4.6378, -3.6638, -4.1768,  0.3203,  4.5009],
#        device='npu:0')

# tensor([[-0.3558,  0.3476,  0.6510,  ..., -0.8799,  0.1510,  0.0872]],
#        device='npu:0')
# tensor([[ 1.1465e-02, -2.6519e-03, -6.2085e-04,  ..., -1.1440e-02,
#          -4.3514e-03,  8.4792e-05],
#         [ 9.2933e-03,  3.7127e-03, -1.3112e-02,  ..., -2.9217e-02,
#           7.5597e-04,  1.6280e-03],
#         [-6.4706e-03, -7.2132e-03,  1.2318e-02,  ...,  2.2557e-02,
#          -2.3140e-03, -2.0004e-03],
#         ...,
#         [ 5.2980e-03,  1.7142e-02,  1.3669e-02,  ..., -3.2009e-02,
#          -3.6876e-03, -1.0002e-03],
#         [-1.4005e-03,  1.2475e-02,  8.7018e-03,  ...,  1.2997e-02,
#           2.8579e-03,  3.1922e-03],
#         [ 1.1019e-03, -5.3251e-03, -1.0490e-02,  ...,  2.8572e-02,
#           1.6318e-03, -5.0277e-04]], device='npu:0')
# tensor([-0.6253, -2.1899, -0.9417,  3.4598, -0.7620, -2.9832, -1.2159,  3.1720,
#         -0.7625, -2.9741, -1.4243,  3.3009, -0.8243, -3.0176, -1.5190,  3.6235],
#        device='npu:0')

# tensor([[ 1.0069,  1.3928,  0.3434,  ..., -0.5003, -0.5441, -0.9864]],
#        device='npu:0')
# tensor([[-0.0324, -0.0106, -0.0003,  ..., -0.0065,  0.0157, -0.0010],
#         [-0.0263,  0.0149, -0.0069,  ..., -0.0166, -0.0027, -0.0184],
#         [ 0.0183, -0.0289,  0.0065,  ...,  0.0128,  0.0083,  0.0226],
#         ...,
#         [-0.0150,  0.0687,  0.0072,  ..., -0.0182,  0.0133,  0.0113],
#         [ 0.0040,  0.0500,  0.0046,  ...,  0.0074, -0.0103, -0.0361],
#         [-0.0031, -0.0213, -0.0055,  ...,  0.0162, -0.0059,  0.0057]],
#        device='npu:0')
# tensor([-5.1604, -5.0711, -0.3484,  6.1214, -5.4565, -4.3618,  0.0531,  6.7870,
#         -4.5066, -3.8735,  1.2001,  6.4662, -5.7796, -3.9636,  1.0662,  6.7369],
#        device='npu:0')

In [26]:
# tensor([[ 0.1855,  0.5378,  0.6770,  ...,  0.1555,  0.6903, -0.5417]],
#        device='npu:0')
# tensor([[-0.0060, -0.0041, -0.0006,  ...,  0.0020, -0.0199, -0.0005],
#         [-0.0048,  0.0057, -0.0136,  ...,  0.0052,  0.0035, -0.0101],
#         [ 0.0034, -0.0112,  0.0128,  ..., -0.0040, -0.0106,  0.0124],
#         ...,
#         [-0.0028,  0.0265,  0.0142,  ...,  0.0057, -0.0169,  0.0062],
#         [ 0.0007,  0.0193,  0.0090,  ..., -0.0023,  0.0131, -0.0198],
#         [-0.0006, -0.0082, -0.0109,  ..., -0.0050,  0.0075,  0.0031]],
#        device='npu:0')
# tensor([-1.3897, -3.9430, -4.3469,  6.1968, -0.8270, -3.7137, -3.3280,  6.0117,
#         -0.5749, -2.7710, -3.3840,  6.4286, -1.5389, -2.4827, -3.5520,  6.0182],
#        device='npu:0')

# tensor([[ 1.0254,  0.5812, -0.6106,  ...,  0.4082,  0.6002, -0.8905]],
#        device='npu:0')
# tensor([[-0.0330, -0.0044,  0.0006,  ...,  0.0053, -0.0173, -0.0009],
#         [-0.0268,  0.0062,  0.0123,  ...,  0.0136,  0.0030, -0.0166],
#         [ 0.0187, -0.0121, -0.0116,  ..., -0.0105, -0.0092,  0.0204],
#         ...,
#         [-0.0153,  0.0287, -0.0128,  ...,  0.0148, -0.0147,  0.0102],
#         [ 0.0040,  0.0209, -0.0082,  ..., -0.0060,  0.0114, -0.0326],
#         [-0.0032, -0.0089,  0.0098,  ..., -0.0133,  0.0065,  0.0051]],
#        device='npu:0')
# tensor([-6.0129,  1.7111, -6.5761,  5.9215, -3.9545,  2.4479, -5.5409,  7.3328,
#         -4.5959,  1.5959, -5.4070,  6.3355, -6.0616,  1.1282, -5.3303,  6.2085],
#        device='npu:0')

# tensor([[ 1.3194, -0.1567, -0.8584,  ...,  0.1426, -0.0675, -0.9956]],
#        device='npu:0')
# tensor([[-0.0425,  0.0012,  0.0008,  ...,  0.0019,  0.0019, -0.0010],
#         [-0.0345, -0.0017,  0.0173,  ...,  0.0047, -0.0003, -0.0186],
#         [ 0.0240,  0.0033, -0.0162,  ..., -0.0037,  0.0010,  0.0228],
#         ...,
#         [-0.0196, -0.0077, -0.0180,  ...,  0.0052,  0.0016,  0.0114],
#         [ 0.0052, -0.0056, -0.0115,  ..., -0.0021, -0.0013, -0.0365],
#         [-0.0041,  0.0024,  0.0138,  ..., -0.0046, -0.0007,  0.0057]],
#        device='npu:0')
# tensor([-6.0175,  3.4365, -4.2462,  5.2776, -5.4512,  2.7193, -5.9480,  3.8372,
#         -6.0735,  2.4740, -5.0653,  5.0374, -5.7462,  2.6191, -6.8862,  4.0672],
#        device='npu:0')

# tensor([[ 1.5842,  1.2670, -0.1690,  ..., -1.0802,  0.5013, -0.9520]],
#        device='npu:0')
# tensor([[-0.0511, -0.0097,  0.0002,  ..., -0.0140, -0.0144, -0.0009],
#         [-0.0414,  0.0135,  0.0034,  ..., -0.0359,  0.0025, -0.0178],
#         [ 0.0288, -0.0263, -0.0032,  ...,  0.0277, -0.0077,  0.0218],
#         ...,
#         [-0.0236,  0.0625, -0.0035,  ..., -0.0393, -0.0122,  0.0109],
#         [ 0.0062,  0.0455, -0.0023,  ...,  0.0160,  0.0095, -0.0349],
#         [-0.0049, -0.0194,  0.0027,  ...,  0.0351,  0.0054,  0.0055]],
#        device='npu:0')
# tensor([-6.1194, -4.5111, -0.6627,  7.2502, -5.7669, -3.8878, -0.7322,  6.4212,
#         -4.8878, -3.6560, -0.1176,  8.0237, -6.1632, -4.2981, -0.1126,  7.0316],
#        device='npu:0')

# tensor([[ 1.0868,  0.3668, -0.2697,  ..., -0.9732, -0.0321, -0.9765]],
#        device='npu:0')
# tensor([[-0.0350, -0.0028,  0.0003,  ..., -0.0127,  0.0009, -0.0009],
#         [-0.0284,  0.0039,  0.0054,  ..., -0.0323, -0.0002, -0.0182],
#         [ 0.0198, -0.0076, -0.0051,  ...,  0.0249,  0.0005,  0.0224],
#         ...,
#         [-0.0162,  0.0181, -0.0057,  ..., -0.0354,  0.0008,  0.0112],
#         [ 0.0043,  0.0132, -0.0036,  ...,  0.0144, -0.0006, -0.0358],
#         [-0.0034, -0.0056,  0.0043,  ...,  0.0316, -0.0003,  0.0056]],
#        device='npu:0')
# tensor([-4.5256, -5.0533, -2.2442,  6.9816, -4.3125, -4.6432, -2.2281,  8.3042,
#         -3.2991, -3.8551, -2.1794,  8.4313, -4.0957, -4.3191, -1.6784,  7.9212],
#        device='npu:0')

# tensor([[ 1.0069,  1.3928,  0.3434,  ..., -0.5003, -0.5441, -0.9864]],
#        device='npu:0')
# tensor([[-0.0324, -0.0106, -0.0003,  ..., -0.0065,  0.0157, -0.0010],
#         [-0.0263,  0.0149, -0.0069,  ..., -0.0166, -0.0027, -0.0184],
#         [ 0.0183, -0.0289,  0.0065,  ...,  0.0128,  0.0083,  0.0226],
#         ...,
#         [-0.0150,  0.0687,  0.0072,  ..., -0.0182,  0.0133,  0.0113],
#         [ 0.0040,  0.0500,  0.0046,  ...,  0.0074, -0.0103, -0.0361],
#         [-0.0031, -0.0213, -0.0055,  ...,  0.0162, -0.0059,  0.0057]],
#        device='npu:0')
# tensor([-5.1604, -5.0711, -0.3484,  6.1214, -5.4565, -4.3618,  0.0531,  6.7870,
#         -4.5066, -3.8735,  1.2001,  6.4662, -5.7796, -3.9636,  1.0662,  6.7369],
#        device='npu:0')

# tensor([[ 1.0216,  0.8529, -0.4388,  ...,  0.0770,  0.5017, -0.4633]],
#        device='npu:0')
# tensor([[-0.0329, -0.0065,  0.0004,  ...,  0.0010, -0.0145, -0.0005],
#         [-0.0267,  0.0091,  0.0088,  ...,  0.0026,  0.0025, -0.0087],
#         [ 0.0186, -0.0177, -0.0083,  ..., -0.0020, -0.0077,  0.0106],
#         ...,
#         [-0.0152,  0.0421, -0.0092,  ...,  0.0028, -0.0122,  0.0053],
#         [ 0.0040,  0.0306, -0.0059,  ..., -0.0011,  0.0095, -0.0170],
#         [-0.0032, -0.0131,  0.0071,  ..., -0.0025,  0.0054,  0.0027]],
#        device='npu:0')
# tensor([-5.1790, -4.4488, -3.1230,  8.9587, -4.3405, -4.5438, -2.7859,  9.6047,
#         -4.6704, -4.0651, -2.3187,  9.5582, -4.9057, -4.2406, -2.0761,  9.7226],
#        device='npu:0')

# tensor([[ 0.1088,  0.5044, -0.1628,  ..., -0.9480,  0.3878, -0.1708]],
#        device='npu:0')
# tensor([[-0.0035, -0.0038,  0.0002,  ..., -0.0123, -0.0112, -0.0002],
#         [-0.0028,  0.0054,  0.0033,  ..., -0.0315,  0.0019, -0.0032],
#         [ 0.0020, -0.0105, -0.0031,  ...,  0.0243, -0.0059,  0.0039],
#         ...,
#         [-0.0016,  0.0249, -0.0034,  ..., -0.0345, -0.0095,  0.0020],
#         [ 0.0004,  0.0181, -0.0022,  ...,  0.0140,  0.0073, -0.0063],
#         [-0.0003, -0.0077,  0.0026,  ...,  0.0308,  0.0042,  0.0010]],
#        device='npu:0')
# tensor([-2.0002, -2.0026, -2.0538,  5.9903, -1.9553, -2.8954, -2.0097,  5.7737,
#         -1.8729, -2.6026, -2.2868,  5.7686, -2.0687, -2.2999, -2.3535,  5.9686],
#        device='npu:0')

In [27]:
# probes = list(range(5400, 5600)) + list(range(6500, 6700))
probes = None
labels = [0, 0, 0, 0]
idx2labels = ['general', 'reason', 'math', 'code']
tasks = [0, 0, 0, 0, 0, 0, 0, 0]
idx2tasks = ['ultrachat', 'cosmos', 'squad', 'gsm8k', 'mathinstruct', 'math', 'leetcode', 'magicoder']
results = {
    "type": {
        "general": 0,
        "reason": 0,
        "math": 0,
        "code": 0,
    },
    "task": {
        "ultrachat":0,
        "cosmos":0,
        "squad":0,
        "gsm8k":0,
        "mathinstruct": 0,
        "math":0,
        "leetcode":0,
        "magicoder":0,
    },
    "loss": {
        "type": {
            "general": 0,
            "reason": 0,
            "math": 0,
            "code": 0,
        },
        "task": {
            "ultrachat":0,
            "cosmos":0,
            "squad":0,
            "gsm8k":0,
            "mathinstruct": 0,
            "math":0,
            "leetcode":0,
            "magicoder":0,
        },
    }
}
code_negative = []
magicoder_negative = []
with torch.no_grad():
    for i, item in enumerate(tqdm(mix_domains_tokens)):
        if probes is not None and i not in probes:
            continue
        outputs = model(
            input_ids=torch.tensor([item['input_ids']], device=model.device),
            labels=torch.tensor([item['labels']], device=model.device),
            type_labels=torch.tensor([item['type']], device=model.device),
            task_labels=torch.tensor([item['task']], device=model.device),
            use_cache=False,
        )
        labels[item['type']] += model.label_block
        tasks[item['task']] += model.task_block

        type_judges = (outputs[0].argmax(dim=-1).cpu() == item['type']).sum().item()
        task_judges = (outputs[1].argmax(dim=-1).cpu() == item['task']).sum().item()
        results['type'][idx2labels[item['type']]] += type_judges
        results['task'][idx2tasks[item['task']]] += task_judges
        results['loss']['type'][idx2labels[item['type']]] += outputs[2].sum()
        results['loss']['task'][idx2tasks[item['task']]] += outputs[3].sum()
        if item['type'] == len(idx2labels) - 1 and type_judges != model.label_block:
            code_negative.append(tokenizer.decode(item['input_ids']))
        if item['task'] == len(idx2tasks) - 1 and task_judges != model.task_block:
            magicoder_negative.append(tokenizer.decode(item['input_ids']))

for i, k in enumerate(idx2labels):
    results['type'][k] = results['type'][k] / labels[i]
for i, k in enumerate(idx2tasks):
    results['task'][k] = results['task'][k] / tasks[i]
for i, k in enumerate(idx2labels):
    results['loss']['type'][k] = model.label_block * results['loss']['type'][k] / labels[i]
for i, k in enumerate(idx2tasks):
    results['loss']['task'][k] = model.task_block * results['loss']['task'][k] / tasks[i]

100%|██████████| 7192/7192 [17:15<00:00,  6.95it/s]


In [28]:
# print(len(code_negative), len(magicoder_negative))
# # print(code_negative[5])
# print(magicoder_negative[40])

In [29]:
print(labels)
print(tasks)
def print_dict(d, indent=0):
    for key, value in d.items():
        if isinstance(value, dict):
            print(" " * indent + f"{key}:")
            print_dict(value, indent + 4)  # 增加缩进
        else:
            print(" " * indent + f"{key}: {value}")

print_dict(results)


[690432, 1380864, 2071296, 1380864]
[345216, 345216, 345216, 345216, 345216, 345216, 345216, 345216]
type:
    general: 0.9998059186132741
    reason: 0.9999869646829811
    math: 0.9964963964590285
    code: 1.0
task:
    ultrachat: 1.0
    cosmos: 1.0
    squad: 1.0
    gsm8k: 1.0
    mathinstruct: 0.9891575129773823
    math: 1.0
    leetcode: 1.0
    magicoder: 1.0
loss:
    type:
        general: 0.08110661059617996
        reason: 0.06692225486040115
        math: 0.06706884503364563
        code: 0.1827438473701477
    task:
        ultrachat: 0.003249982139095664
        cosmos: 0.02484956383705139
        squad: 0.024770913645625114
        gsm8k: 0.04492901265621185
        mathinstruct: 0.08342557400465012
        math: 0.050189059227705
        leetcode: 0.19217252731323242
        magicoder: 0.22151687741279602


In [30]:
# type:
#     general: 0.5941352660363367
#     reason: 0.9843677581572117
#     math: 0.9560844997528117
#     code: 0.9019099636169818
# task:
#     ultrachat: 0.7077684695958473
#     cosmos: 0.9934302002224694
#     squad: 0.9519112671486837
#     gsm8k: 0.9834248702261772
#     mathinstruct: 0.7118673526140156
#     math: 0.9865678299962922
#     leetcode: 0.9563809325176121
#     magicoder: 0.7513411892843901
# loss:
#     type:
#         general: 1.1336313486099243
#         reason: 0.3568684756755829
#         math: 0.46383780241012573
#         code: 0.6989308595657349
#     task:
#         ultrachat: 1.3710956573486328
#         cosmos: 0.3850761950016022
#         squad: 0.5187223553657532
#         gsm8k: 0.6053699254989624
#         mathinstruct: 1.1713249683380127
#         math: 0.5155571699142456
#         leetcode: 0.7877065539360046
#         magicoder: 1.2719649076461792

In [31]:
# type:
#     general: 0.8171188473303671
#     reason: 0.9951877954671857
#     math: 0.9659498207885304
#     code: 0.9732703582684464
# task:
#     ultrachat: 0.9086484983314794
#     cosmos: 0.9997450871338525
#     squad: 0.9824370828698554
#     gsm8k: 0.9993018863552094
#     mathinstruct: 0.6453090239154616
#     math: 0.9992989896180942
#     leetcode: 0.9571340841675936
#     magicoder: 0.8840928578049685
# loss:
#     type:
#         general: 0.8247817158699036
#         reason: 0.23958292603492737
#         math: 0.3445204794406891
#         code: 0.4357350170612335
#     task:
#         ultrachat: 0.7213690280914307
#         cosmos: 0.21618574857711792
#         squad: 0.29625970125198364
#         gsm8k: 0.4098522663116455
#         mathinstruct: 1.1527856588363647
#         math: 0.34112346172332764
#         leetcode: 0.6382291316986084
#         magicoder: 0.797913134098053

In [32]:
# type:
#     general: 0.2922967069892473
#     reason: 0.9985342510196514
#     math: 0.8445977783957483
#     code: 0.8177597504171301
# task:
#     ultrachat: 0.451386957730812
#     cosmos: 0.9893371106785317
#     squad: 0.9911128105302187
#     gsm8k: 0.9332070355951056
#     mathinstruct: 0.5675692899517983
#     math: 0.9604508481646273
#     leetcode: 0.9810987903225806
#     magicoder: 0.4952841119762699
# loss:
#     type:
#         general: 1.4525173902511597
#         reason: 0.12175177037715912
#         math: 0.6823084950447083
#         code: 0.7812681794166565
#     task:
#         ultrachat: 1.6798996925354004
#         cosmos: 0.2027428150177002
#         squad: 0.157377690076828
#         gsm8k: 0.8757413625717163
#         mathinstruct: 1.438187599182129
#         math: 0.739412784576416
#         leetcode: 0.655958354473114
#         magicoder: 1.5841748714447021

In [33]:
# type:
#     general: 0.46200205088987767
#     reason: 0.9999840679458658
#     math: 0.9098926469225065
#     code: 0.9605601999907304
# task:
#     ultrachat: 0.8196781145717463
#     cosmos: 0.9996697719688543
#     squad: 0.9999855163144235
#     gsm8k: 0.9954260520949203
#     mathinstruct: 0.5528712458286985
#     math: 0.996043057100482
#     leetcode: 0.9962950732295143
#     magicoder: 0.6171034946236559
# loss:
#     type:
#         general: 1.2567133903503418
#         reason: 0.04493677243590355
#         math: 0.5674847960472107
#         code: 0.49269017577171326
#     task:
#         ultrachat: 1.0468330383300781
#         cosmos: 0.07551289349794388
#         squad: 0.029741195961833
#         gsm8k: 0.6640282869338989
#         mathinstruct: 1.4108738899230957
#         math: 0.5190497636795044
#         leetcode: 0.4250815212726593
#         magicoder: 1.2243057489395142

In [34]:
# type:
#     general: 0.0
#     reason: 0.0015859635706340378
#     math: 8.690211345939933e-06
#     code: 1.0
# task:
#     ultrachat: 0.0
#     cosmos: 0.0
#     squad: 0.00048013417686318133
#     gsm8k: 0.0
#     mathinstruct: 0.0
#     math: 0.0
#     leetcode: 0.0
#     magicoder: 1.0
# loss:
#     type:
#         general: 4.285716533660889
#         reason: 4.467981338500977
#         math: 4.76007080078125
#         code: 0.018939977511763573
#     task:
#         ultrachat: 6.664792537689209
#         cosmos: 8.158964157104492
#         squad: 5.784264087677002
#         gsm8k: 7.78471565246582
#         mathinstruct: 7.059243679046631
#         math: 6.677797317504883
#         leetcode: 4.487239837646484
#         magicoder: 0.012047037482261658

# type:
#     general: 0.3109183815350389
#     reason: 0.9988018371106785
#     math: 0.8777519002595476
#     code: 0.8343124304783093
# task:
#     ultrachat: 0.5299138800055617
#     cosmos: 0.9939364050333704
#     squad: 0.9941666956340378
#     gsm8k: 0.9737751147107898
#     mathinstruct: 0.6578381361234705
#     math: 0.9741009976362626
#     leetcode: 0.98588926932703
#     magicoder: 0.5799673248053393
# loss:
#     type:
#         general: 1.4366763830184937
#         reason: 0.10414722561836243
#         math: 0.6093225479125977
#         code: 0.7424556612968445
#     task:
#         ultrachat: 1.5710420608520508
#         cosmos: 0.1614801287651062
#         squad: 0.13074365258216858
#         gsm8k: 0.6924129724502563
#         mathinstruct: 1.2668626308441162
#         math: 0.6172947287559509
#         leetcode: 0.55473792552948
#         magicoder: 1.4595506191253662

In [35]:
# type:
#     general: 0.5123270647942157
#     reason: 0.9999945686179088
#     math: 0.9341057482851316
#     code: 0.9646015103587319
# task:
#     ultrachat: 0.8635245759176863
#     cosmos: 0.9998913723581757
#     squad: 0.9999956548943271
#     gsm8k: 0.9986703976640712
#     mathinstruct: 0.6287476536429366
#     math: 0.9981381222191323
#     leetcode: 0.9961350285038932
#     magicoder: 0.6784469723303671
# loss:
#     type:
#         general: 1.2010513544082642
#         reason: 0.03792679309844971
#         math: 0.4856143593788147
#         code: 0.45562025904655457
#     task:
#         ultrachat: 0.8995531797409058
#         cosmos: 0.05838460475206375
#         squad: 0.02379104122519493
#         gsm8k: 0.48651596903800964
#         mathinstruct: 1.2296355962753296
#         math: 0.4172775149345398
#         leetcode: 0.35403645038604736
#         magicoder: 1.0919538736343384

In [36]:
# [460288, 920576, 1380864, 920576]
# [460288, 460288, 460288, 460288, 460288, 460288, 460288, 460288]
# type:
#     general: 0.07636523220244716
#     reason: 0.7209757803809789
#     math: 0.20362541133667036
#     code: 0.948788584538376
# task:
#     ultrachat: 0.07011045258620689
#     cosmos: 0.6401014147664071
#     squad: 0.8917938334260289
#     gsm8k: 0.12254936040044494
#     mathinstruct: 0.12804374652391545
#     math: 0.3662750278086763
#     leetcode: 0.6794789349276974
#     magicoder: 0.700087771134594
# loss:
#     type:
#         general: 1.7394603490829468
#         reason: 0.7293990254402161
#         math: 1.4539313316345215
#         code: 0.556153416633606
#     task:
#         ultrachat: 2.26645565032959
#         cosmos: 1.2400052547454834
#         squad: 0.673072874546051
#         gsm8k: 2.085217237472534
#         mathinstruct: 2.148524761199951
#         math: 1.6924052238464355
#         leetcode: 1.3344554901123047
#         magicoder: 1.413692593574524

# [460288, 920576, 1380864, 920576]
# [460288, 460288, 460288, 460288, 460288, 460288, 460288, 460288]
# type:
#     general: 0.07572650166852057
#     reason: 0.2834996784621802
#     math: 0.13266331803856135
#     code: 0.5357102509733037
# task:
#     ultrachat: 0.0385997462458287
#     cosmos: 0.10468880353170189
#     squad: 0.20679226918798665
#     gsm8k: 0.042221391824249166
#     mathinstruct: 0.04010967046718576
#     math: 0.06190906562847608
#     leetcode: 0.052569260984427144
#     magicoder: 0.5212062882369299
# loss:
#     type:
#         general: 2.0006470680236816
#         reason: 1.5516080856323242
#         math: 1.8972818851470947
#         code: 1.144525170326233
#     task:
#         ultrachat: 2.583346128463745
#         cosmos: 2.413771867752075
#         squad: 2.125948429107666
#         gsm8k: 2.6144256591796875
#         mathinstruct: 2.67024564743042
#         math: 2.5200324058532715
#         leetcode: 2.6003689765930176
#         magicoder: 1.4978017807006836

In [37]:
# [460288, 920576, 1380864, 920576]
# [460288, 460288, 460288, 460288, 460288, 460288, 460288, 460288]
# type:
#     general: 0.3019848442714127
#     reason: 0.9999782744716351
#     math: 0.8905554783092324
#     code: 0.943243143423248
# task:
#     ultrachat: 0.40906345592324805
#     cosmos: 0.9939994090656284
#     squad: 0.9997153955784205
#     gsm8k: 0.7674217011957731
#     mathinstruct: 0.2771590830088988
#     math: 0.9174755805061179
#     leetcode: 0.9928501286151279
#     magicoder: 0.16065376459955505
# loss:
#     type:
#         general: 1.4297796487808228
#         reason: 0.0569545216858387
#         math: 0.6337341070175171
#         code: 0.5854947566986084
#     task:
#         ultrachat: 1.7110114097595215
#         cosmos: 0.2633109390735626
#         squad: 0.08069954812526703
#         gsm8k: 1.3130732774734497
#         mathinstruct: 1.8678638935089111
#         math: 1.1263335943222046
#         leetcode: 0.7887478470802307
#         magicoder: 1.9214529991149902

In [38]:
# [3596, 7192, 10788, 7192]
# [3596, 3596, 3596, 3596, 3596, 3596, 3596, 3596]
# type:
#     general: 0.00027808676307007786
#     reason: 0.19911012235817574
#     math: 0.028735632183908046
#     code: 1.0
# task:
#     ultrachat: 0.0
#     cosmos: 0.015016685205784204
#     squad: 0.27836484983314796
#     gsm8k: 0.0
#     mathinstruct: 0.0011123470522803114
#     math: 0.026418242491657397
#     leetcode: 0.0
#     magicoder: 1.0
# loss:
#     type:
#         general: 4.14914608001709
#         reason: 3.2769949436187744
#         math: 4.309334754943848
#         code: 0.012166154570877552
#     task:
#         ultrachat: 6.074626445770264
#         cosmos: 5.2770795822143555
#         squad: 2.8696818351745605
#         gsm8k: 6.412508487701416
#         mathinstruct: 7.050684928894043
#         math: 4.383765697479248
#         leetcode: 5.4081807136535645
#         magicoder: 0.00883562583476305

# type:
#     general: 0.9290878754171301
#     reason: 1.0
#     math: 0.9734890619206525
#     code: 0.9899888765294772
# task:
#     ultrachat: 0.9340934371523916
#     cosmos: 1.0
#     squad: 1.0
#     gsm8k: 1.0
#     mathinstruct: 0.807285873192436
#     math: 1.0
#     leetcode: 0.9897107897664071
#     magicoder: 0.9479977753058955
# loss:
#     type:
#         general: 0.24478113651275635
#         reason: 0.0015300975646823645
#         math: 0.09646623581647873
#         code: 0.060816794633865356
#     task:
#         ultrachat: 0.24045133590698242
#         cosmos: 0.002225873526185751
#         squad: 0.0019729940686374903
#         gsm8k: 0.03333236649632454
#         mathinstruct: 0.5468396544456482
#         math: 0.05881090089678764
#         leetcode: 0.09744806587696075
#         magicoder: 0.25248274207115173

In [39]:
# type:
#     general: 0.9580088987764183
#     reason: 0.9997219132369299
#     math: 0.9944382647385984
#     code: 0.939238042269188
# task:
#     ultrachat: 0.9657953281423804
#     cosmos: 1.0
#     squad: 0.9980533926585095
#     gsm8k: 0.9997219132369299
#     mathinstruct: 0.8145161290322581
#     math: 0.9994438264738599
#     leetcode: 0.9994438264738599
#     magicoder: 0.7942157953281423
# loss:
#     type:
#         general: 0.13791176676750183
#         reason: 0.006823231466114521
#         math: 0.04296460747718811
#         code: 0.2444450557231903
#     task:
#         ultrachat: 0.14313799142837524
#         cosmos: 0.008301472291350365
#         squad: 0.0161783155053854
#         gsm8k: 0.04754825308918953
#         mathinstruct: 0.4992556869983673
#         math: 0.08070715516805649
#         leetcode: 0.06493852287530899
#         magicoder: 0.6964846849441528

# [3596, 7192, 10788, 7192]
# [3596, 3596, 3596, 3596, 3596, 3596, 3596, 3596]

# type:
#     general: 0.7171857619577308
#     reason: 1.0
#     math: 0.9745087133852428
#     code: 0.9867908787541713
# task:
#     ultrachat: 0.6548943270300334
#     cosmos: 1.0
#     squad: 0.9933259176863182
#     gsm8k: 1.0
#     mathinstruct: 0.639321468298109
#     math: 0.9980533926585095
#     leetcode: 0.996662958843159
#     magicoder: 0.8567853170189099

# type:
#     general: 0.703559510567297
#     reason: 1.0
#     math: 0.9870226177233964
#     code: 0.9803948832035595
# task:
#     ultrachat: 0.6724137931034483
#     cosmos: 1.0
#     squad: 0.9969410456062291
#     gsm8k: 1.0
#     mathinstruct: 0.7372080088987765
#     math: 1.0
#     leetcode: 0.993047830923248
#     magicoder: 0.9221357063403782

# type:
#     general: 0.768075639599555
#     reason: 1.0
#     math: 0.9900815721171672
#     code: 0.9777530589543938
# task:
#     ultrachat: 0.7780867630700778
#     cosmos: 1.0
#     squad: 0.9997219132369299
#     gsm8k: 0.9994438264738599
#     mathinstruct: 0.8125695216907676
#     math: 1.0
#     leetcode: 0.9941601779755284
#     magicoder: 0.9124026696329255

# type:
#     general: 0.9390989988876529
#     reason: 1.0
#     math: 0.9908231368186874
#     code: 0.9917964404894327
# task:
#     ultrachat: 0.9416017797552837
#     cosmos: 1.0
#     squad: 1.0
#     gsm8k: 1.0
#     mathinstruct: 0.7989432703003337
#     math: 1.0
#     leetcode: 0.9927697441601779
#     magicoder: 0.9621802002224694
# loss:
#     type:
#         general: 0.19234523177146912
#         reason: 0.0006273461622186005
#         math: 0.04456143081188202
#         code: 0.05012359470129013
#     task:
#         ultrachat: 0.1941305249929428
#         cosmos: 0.0006491461535915732
#         squad: 0.0007586654392071068
#         gsm8k: 0.018461808562278748
#         mathinstruct: 0.5467557907104492
#         math: 0.04869170859456062
#         leetcode: 0.0834423303604126
#         magicoder: 0.18554279208183289